# 01 — Data Split

`gold.match_features` is the directional match-level training table: two
rows per physical match, keyed by `(match_id, player_id)` and grouped by
the immutable `match_id`. Each row carries the full balanced feature set
(player + opponent rolling stats, differentials, context) in that row's
player perspective, and `match_won` is the label relative to that row's
`player_id` side. Splits are made at physical-match granularity so both
orientations of a match always land in the same band.

In [ ]:
from src.utils import load_env

load_env()

from src.constants import (
    CV_FOLDS,
    DATA_PROCESSED,
    GOLD_MATCHES_TABLE,
    GOLD_PROFILES_TABLE,
    TEST_FRACTION,
    TRAIN_FRACTION,
    VAL_FRACTION,
)

gold_table = GOLD_MATCHES_TABLE

profiles_table = GOLD_PROFILES_TABLE

random_state = 42

cutoff_date = None  # default: computed from data in the split cell; override via papermill

val_cutoff_date = None  # default: computed from data in the split cell; override via papermill

output_dir = str(DATA_PROCESSED)

In [2]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from src.db.training import to_dataframe
from src.features.columns import (
    FEATURE_COLS,
)

Path(output_dir).mkdir(parents=True, exist_ok=True)

In [3]:
print("Loading gold features...")

df = to_dataframe(f"SELECT * FROM {gold_table} ORDER BY match_date, match_id, player_id")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Date range: {df['match_date'].min()} to {df['match_date'].max()}")

In [4]:
# gold.match_features is the directional match-level training table: two
# rows per physical match, keyed (match_id, player_id) and grouped by the
# immutable match_id. match_won is the label relative to that row's
# player_id side (vs its opponent_id).

matches = df
print(f"Rows: {len(matches)} ({matches['match_id'].nunique()} matches)")

In [10]:
# Train/validation/test are chronological bands of the data's date range
# (defaults: 90/5/5; override via papermill-supplied val_cutoff_date /
# cutoff_date). The validation band drives Optuna/early stopping/model
# selection; the test band is the final evaluation split and is never
# consumed by tuning, early stopping, or base-model selection.
#
# Splits are made at physical-match granularity: gold holds TWO directional
# rows per match_id (one per player), and both orientations must land in the
# same band. Both rows share the same match_date, so the date-range cutoffs
# already keep a match together; the explicit guard below asserts that no
# match_id straddles a cutoff.
#
# The fractions close the span exactly (TRAIN+VAL+TEST = 1), so the
# derived contiguous cutoffs partition the whole date range.
assert abs(TRAIN_FRACTION + VAL_FRACTION + TEST_FRACTION - 1.0) < 1e-9
span = df["match_date"].max() - df["match_date"].min()
val_cutoff_date = (
    val_cutoff_date
    if val_cutoff_date is not None
    else df["match_date"].min() + span * (1 - VAL_FRACTION - TEST_FRACTION)
)
cutoff_date = (
    cutoff_date if cutoff_date is not None else df["match_date"].min() + span * (1 - TEST_FRACTION)
)

train = matches[matches["match_date"] < val_cutoff_date]
val = matches[(matches["match_date"] >= val_cutoff_date) & (matches["match_date"] < cutoff_date)]
test = matches[matches["match_date"] >= cutoff_date]
for _name, _split in (("train", train), ("val", val), ("test", test)):
    assert not _split.empty, f"{_name} split is empty; adjust the chronological cutoffs"

# Physical-match guard: no match_id may appear in more than one band.
train_ids, val_ids, test_ids = (set(_split["match_id"]) for _split in (train, val, test))
assert train_ids.isdisjoint(val_ids), "match_id straddles the train/val cutoff"
assert train_ids.isdisjoint(test_ids), "match_id straddles the train/test cutoff"
assert val_ids.isdisjoint(test_ids), "match_id straddles the val/test cutoff"

# Deterministic artifact ordering: match date, then the directional row
# identity (match_id, player_id), so every saved parquet is stable across
# reruns.
train = train.sort_values(["match_date", "match_id", "player_id"]).reset_index(drop=True)
val = val.sort_values(["match_date", "match_id", "player_id"]).reset_index(drop=True)
test = test.sort_values(["match_date", "match_id", "player_id"]).reset_index(drop=True)

X_train, y_train = train[FEATURE_COLS].copy(), train["match_won"]
X_val, y_val = val[FEATURE_COLS].copy(), val["match_won"]
X_test, y_test = test[FEATURE_COLS].copy(), test["match_won"]

# Minimal inference inputs required to rebuild each held-out row, including
# the directional row identity (match_id, player_id) and the physical-match
# group match_id: surface, tournament/round context, indoor state.
info_cols = [
    "match_id",
    "match_date",
    "player_id",
    "opponent_id",
    "surface",
    "tournament",
    "round",
    "tournament_level",
    "round_encoded",
    "is_indoor",
]

# The finalized gold contract: every FEATURE_COLS cell is already
# non-null and finite (dbt + snapshot validation enforce this). No
# imputer is fitted here; the split asserts the contract so training
# can never read NULL/NaN/Infinity feature values.
for _name, _split in (("train", X_train), ("val", X_val), ("test", X_test)):
    _values = np.asarray(_split, dtype=float)
    assert not np.isnan(_values).any(), f"gold FEATURE_COLS contains NaN in {_name}"
    assert np.isfinite(_values).all(), f"gold FEATURE_COLS non-finite in {_name}"

print(f"Training set: {len(train)} rows ({train['match_id'].nunique()} matches)")
print(f"Validation set: {len(val)} rows ({val['match_id'].nunique()} matches)")
print(f"Test set: {len(test)} rows ({test['match_id'].nunique()} matches)")

In [6]:
# ── Save ──
pd.DataFrame({"y": y_train}).to_parquet(f"{output_dir}/y_train.parquet", index=False)
pd.DataFrame({"y": y_val}).to_parquet(f"{output_dir}/y_val.parquet", index=False)
pd.DataFrame({"y": y_test}).to_parquet(f"{output_dir}/y_test.parquet", index=False)

for name, data in [
    ("X_train", X_train),
    ("X_val", X_val),
    ("X_test", X_test),
    ("info_train", train[info_cols]),
    ("info_val", val[info_cols]),
    ("info_test", test[info_cols]),
]:
    path = f"{output_dir}/{name}.parquet"
    data.to_parquet(path)
    print(f"Saved {path} ({len(data)} rows)")

# Max match date across every split (train, validation, test); evaluation
# rejects the candidate before registration when this is later than the
# current UTC date.
split_meta = {
    "max_match_date": str(df["match_date"].max().date()),
    "n_train": len(train),
    "n_train_matches": int(train["match_id"].nunique()),
    "n_val": len(val),
    "n_val_matches": int(val["match_id"].nunique()),
    "n_test": len(test),
    "n_test_matches": int(test["match_id"].nunique()),
    "val_cutoff_date": str(val_cutoff_date.date()),
    "test_cutoff_date": str(cutoff_date.date()),
}
with open(f"{output_dir}/split_meta.json", "w") as f:
    json.dump(split_meta, f, indent=2)
print(f"Max match date across all splits: {split_meta['max_match_date']}")

In [ ]:
# ── Grouped-CV fold assignment for this run's train split ──
# Created exactly once per training run, immediately after the new split
# artifacts are written. Folds are time-forward date bands of the train
# split: fold k validates band k+1 and trains on strictly earlier dates
# (band 0 is grow-in data, never validated). match_won is passed so the
# directional-label invariant is validated before persisting. Each 02
# tuner loads and validates this file against its own info_train; no 02
# notebook ever writes it.
from src.models.grouped_cv import create_fold_assignment

fold_frame = create_fold_assignment(
    train["match_id"],
    train["match_date"],
    CV_FOLDS,
    random_state,
    train["match_won"],
    f"{output_dir}/fold_assignment.parquet",
)
print(
    f"Saved {output_dir}/fold_assignment.parquet "
    f"({len(fold_frame)} train matches across {CV_FOLDS} folds)"
)